# Pipeline de Análise Exploratória com spark_eda

Este notebook demonstra o pipeline completo de **Análise Exploratória de Dados (EDA)** 
e **Avaliação de Qualidade** usando a biblioteca `spark_eda`.

### Fluxo do Pipeline:
1. **Geração de Dados** — Criação de um DataFrame PySpark complexo (100.000 registros)
2. **Análise Completa** — `spark_eda.analyze()` → relatório com 9 seções
3. **Avaliação de Qualidade** — `spark_eda.assess_quality()` → score e penalizadores
4. **Exportação** — Geração de relatório HTML para visualização

---

## 1. Configuração Inicial

In [6]:
from __future__ import annotations

import time
from datetime import datetime
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

import spark_eda
from spark_eda import EDAConfig, QualityConfig

print(f"spark_eda version: {spark_eda.__version__}")

spark_eda version: 0.1.0


### 1.1 Criar SparkSession

In [7]:
spark = (
    SparkSession.builder.master("local[*]")
    .appName("spark-eda-pipeline")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

print(f"SparkSession pronta: {spark}")
print(f"Spark version: {spark.version}")

SparkSession pronta: <pyspark.sql.session.SparkSession object at 0x79a8183e3920>
Spark version: 4.2.0


## 2. Geração de Dados

Criamos um DataFrame complexo com **100.000 registros** e 18 colunas de tipos variados:
- Numéricas: `id`, `idade`, `salario`, `score`, `dependentes`, `tempo_casa_dias`
- Categóricas: `cidade`, `estado`, `categoria`, `regiao`
- Texto: `nome`, `email`, `documento`, `cep`, `observacao`
- Temporal: `data_cadastro`, `ultimo_acesso`
- Booleana: `ativo`
- Edge cases: `coluna_constante`, `coluna_quase_constante`, `timestamp_nulo`

In [8]:
CIDADES_BRASIL = [
    ("São Paulo", "SP", "Sudeste"),
    ("Rio de Janeiro", "RJ", "Sudeste"),
    ("Belo Horizonte", "MG", "Sudeste"),
    ("Salvador", "BA", "Nordeste"),
    ("Fortaleza", "CE", "Nordeste"),
    ("Recife", "PE", "Nordeste"),
    ("Brasília", "DF", "Centro-Oeste"),
    ("Curitiba", "PR", "Sul"),
    ("Porto Alegre", "RS", "Sul"),
    ("Manaus", "AM", "Norte"),
    ("Belém", "PA", "Norte"),
    ("Goiânia", "GO", "Centro-Oeste"),
    ("Campinas", "SP", "Sudeste"),
    ("São Luís", "MA", "Nordeste"),
    ("Maceió", "AL", "Nordeste"),
    ("Natal", "RN", "Nordeste"),
    ("Teresina", "PI", "Nordeste"),
    ("João Pessoa", "PB", "Nordeste"),
    ("Aracaju", "SE", "Nordeste"),
    ("Cuiabá", "MT", "Centro-Oeste"),
]
CATEGORIAS = [
    "Eletrônicos",
    "Roupas",
    "Alimentos",
    "Livros",
    "Esportes",
    "Beleza",
    "Casa",
    "Automotivo",
    "Brinquedos",
    "Saúde",
]
NUM_CIDADES = len(CIDADES_BRASIL)
NUM_CATEGORIAS = len(CATEGORIAS)

NUM_ROWS = 100_000

In [9]:
# --- Expressões when para lookup distribuído ---
cidade_expr = F.when(F.rand() < 0.05, None)
estado_expr = F.when(F.rand() < 0.05, None)
regiao_expr = F.when(F.rand() < 0.05, None)
for i, (cid, est, reg) in enumerate(CIDADES_BRASIL):
    cond = F.col("id") % NUM_CIDADES == i
    cidade_expr = cidade_expr.when(cond, F.lit(cid))
    estado_expr = estado_expr.when(cond, F.lit(est))
    regiao_expr = regiao_expr.when(cond, F.lit(reg))
cidade_expr = cidade_expr.otherwise(F.lit(CIDADES_BRASIL[0][0]))
estado_expr = estado_expr.otherwise(F.lit(CIDADES_BRASIL[0][1]))
regiao_expr = regiao_expr.otherwise(F.lit(CIDADES_BRASIL[0][2]))

cat_expr = F.when(F.rand() < 0.03, None)
for i, cat in enumerate(CATEGORIAS):
    cat_expr = cat_expr.when(F.col("id") % NUM_CATEGORIAS == i, F.lit(cat))
cat_expr = cat_expr.otherwise(F.lit(CATEGORIAS[0]))

# --- DataFrame base ---
print(f"Gerando {NUM_ROWS:,} registros...")
start_gen = time.time()

base = spark.range(0, NUM_ROWS, 1, numPartitions=8)

df = base.select(
    F.col("id"),
    F.concat(
        F.lit("Pessoa_"), F.when(F.rand() < 0.05, None).otherwise(F.expr("printf('%06d', CAST(rand() * 50000 AS INT))"))
    ).alias("nome"),
    F.when(F.rand() < 0.03, None)
    .otherwise(
        F.concat(
            F.lit("user"),
            F.expr("printf('%04d', CAST(rand() * 90000 AS INT))"),
            F.lit("@exemplo.com.br"),
        )
    )
    .alias("email"),
    cidade_expr.alias("cidade"),
    estado_expr.alias("estado"),
    F.when(F.rand() < 0.02, None)
    .otherwise(
        F.greatest(
            F.lit(18),
            F.least(F.lit(80), (F.randn() * 12 + 38).cast("int")),
        )
    )
    .alias("idade"),
    F.when(F.rand() < 0.06, None)
    .otherwise(
        F.round(
            F.least(F.lit(35000.0), F.greatest(F.lit(1200.0), F.exp(F.randn() * 0.8 + 8.5))),
            2,
        )
    )
    .alias("salario"),
    F.round(F.rand() * 1000, 2).alias("score"),
    cat_expr.alias("categoria"),
    F.date_add(F.lit("2019-01-01"), (F.rand() * 2191).cast("int")).alias("data_cadastro"),
    F.when(F.rand() < 0.08, None)
    .otherwise(
        F.timestamp_seconds(
            F.lit(1577836800) + (F.rand() * 157766400).cast("int"),
        )
    )
    .alias("ultimo_acesso"),
    (F.rand() < 0.8).alias("ativo"),
    F.when(F.rand() < 0.01, None)
    .otherwise(F.least(F.lit(5), F.greatest(F.lit(0), (F.abs(F.randn()) * 1.5).cast("int"))))
    .alias("dependentes"),
    (F.rand() * 2191).cast("int").alias("tempo_casa_dias"),
    F.when(F.rand() < 0.02, None)
    .otherwise(F.expr("printf('%011d', CAST(rand() * 99999999999 AS LONG))"))
    .alias("documento"),
    F.concat(
        F.expr("printf('%05d', CAST(rand() * 99999 AS INT))"),
        F.lit("-"),
        F.expr("printf('%03d', CAST(rand() * 999 AS INT))"),
    ).alias("cep"),
    F.when(F.rand() < 0.01, F.lit(""))
    .otherwise(F.expr("substr(sha2(cast(rand() as string), 256), 1, CAST(rand() * 100 + 10 AS INT))"))
    .alias("observacao"),
    regiao_expr.alias("regiao"),
)

# Adicionar edge cases
df = df.withColumn("coluna_constante", F.lit("valor_imutavel"))
df = df.withColumn(
    "coluna_quase_constante",
    F.when(F.rand() < 0.001, F.expr("printf('%04d', CAST(rand() * 1000 AS INT))")).otherwise(F.lit("predominante")),
)
df = df.withColumn("timestamp_nulo", F.lit(None).cast("timestamp"))

elapsed_gen = time.time() - start_gen
print(f"DataFrame gerado em {elapsed_gen:.2f}s")
print(f"Schema: {df.schema}")
print(f"Total de registros: {df.count():,}")

Gerando 100,000 registros...
DataFrame gerado em 0.60s
Schema: StructType([StructField('id', LongType(), False), StructField('nome', StringType(), True), StructField('email', StringType(), True), StructField('cidade', StringType(), True), StructField('estado', StringType(), True), StructField('idade', IntegerType(), True), StructField('salario', DoubleType(), True), StructField('score', DoubleType(), True), StructField('categoria', StringType(), True), StructField('data_cadastro', DateType(), True), StructField('ultimo_acesso', TimestampType(), True), StructField('ativo', BooleanType(), False), StructField('dependentes', IntegerType(), True), StructField('tempo_casa_dias', IntegerType(), True), StructField('documento', StringType(), True), StructField('cep', StringType(), False), StructField('observacao', StringType(), True), StructField('regiao', StringType(), True), StructField('coluna_constante', StringType(), False), StructField('coluna_quase_constante', StringType(), False), Struc

### 2.1 Amostra dos Dados

In [10]:
print("\nPrimeiras 10 linhas:")
df.show(10, truncate=40)

print("\nResumo estatístico rápido:")
df.describe().show()


Primeiras 10 linhas:
+---+-------------+------------------------+--------------+------+-----+--------+------+-----------+-------------+-------------------+-----+-----------+---------------+-----------+---------+----------------------------------------+------------+----------------+----------------------+--------------+
| id|         nome|                   email|        cidade|estado|idade| salario| score|  categoria|data_cadastro|      ultimo_acesso|ativo|dependentes|tempo_casa_dias|  documento|      cep|                              observacao|      regiao|coluna_constante|coluna_quase_constante|timestamp_nulo|
+---+-------------+------------------------+--------------+------+-----+--------+------+-----------+-------------+-------------------+-----+-----------+---------------+-----------+---------+----------------------------------------+------------+----------------+----------------------+--------------+
|  0|Pessoa_005796|user85183@exemplo.com.br|     São Paulo|    SP|   45| 2796.

[Stage 11:====================================>                     (5 + 3) / 8]

+-------+-----------------+-------------+--------------------+--------+------+------------------+-----------------+-----------------+---------+------------------+-----------------+--------------------+---------+--------------------+------------+----------------+----------------------+
|summary|               id|         nome|               email|  cidade|estado|             idade|          salario|            score|categoria|       dependentes|  tempo_casa_dias|           documento|      cep|          observacao|      regiao|coluna_constante|coluna_quase_constante|
+-------+-----------------+-------------+--------------------+--------+------+------------------+-----------------+-----------------+---------+------------------+-----------------+--------------------+---------+--------------------+------------+----------------+----------------------+
|  count|           100000|        94929|               96939|   94999| 95015|             97978|            94019|           100000|    96905

## 3. Análise Exploratória Completa

Executamos `spark_eda.analyze()` para gerar um relatório completo com:
- **Overview** (visão geral)
- **Schema** (esquema com tipos inferidos)
- **Quality** (score de qualidade)
- **Stats** (estatísticas descritivas)
- **Distributions** (distribuições)
- **Correlations** (correlações)
- **Outliers** (outliers detectados)
- **Insights** (insights gerados)
- **Recommendations** (recomendações)

In [11]:
print("=" * 60)
print("EXECUTANDO ANÁLISE EXPLORATÓRIA COMPLETA")
print("=" * 60)
print()

config = EDAConfig(
    max_categories=20,
    correlation_methods=("pearson", "cramers_v"),
    outlier_method="iqr",
    enable_insights=True,
    enable_recommendations=True,
    sampling_threshold=500_000,
)

start = time.time()
report = spark_eda.analyze(df, config=config)
elapsed = time.time() - start

print(f"Análise concluída em {elapsed:.2f}s")
print()

EXECUTANDO ANÁLISE EXPLORATÓRIA COMPLETA



Análise concluída em 39.34s



### 3.1 Overview

In [12]:
print(report.overview)
print()
print("HTML preview disponível em Jupyter: report.overview")

Overview
----------------------------------------
  Rows                         100.000
  Columns                           21
  Duplicates                    0 (0,0%)
  Missing                         6,9%
  Est. Size                   59,03 MB

HTML preview disponível em Jupyter: report.overview


### 3.2 Schema

In [13]:
print(report.schema)

Schema
--------------------
+------------------------+-----------+----------+----------------+--------+
| Column                 | Type      | Nullable | Inferred       |  Nulls |
+------------------------+-----------+----------+----------------+--------+
| id                     | long      | No       | auto_increment |      0 |
| nome                   | string    | Yes | —              |   5071 |
| email                  | string    | Yes | email          |   3061 |
| cidade                 | string    | Yes | —              |   5001 |
| estado                 | string    | Yes | —              |   4985 |
| idade                  | integer   | Yes | —              |   2022 |
| salario                | double    | Yes | —              |   5981 |
| score                  | double    | Yes | —              |      0 |
| categoria              | string    | Yes | —              |   3095 |
| data_cadastro          | date      | Yes | —              |      0 |
| ultimo_acesso          | ti

### 3.3 Qualidade dos Dados

In [14]:
quality = report.quality
print(f"Score Geral de Qualidade: {quality.overall:.2f} / 100.0")
print()

for dim in quality.dimensions:
    print(f"  {dim.name.capitalize()}: {dim.score:.2f} (peso: {dim.weight:.2f})")
    for factor in dim.factors[:3]:  # Top 3 fatores
        print(f"    - {factor.name}: {factor.score:.3f} [{factor.severity}]")
        print(f"      {factor.reason}")
print()

print("Top Penalizadores:")
for p in quality.top_penalizers[:5]:
    print(f"  -{p.score * 0.25:.1f} pontos: {p.reason}")
    print(f"    Colunas afetadas: {', '.join(p.affected_columns)}")

Score Geral de Qualidade: 78.18 / 100.0

  Accuracy: 92.87 (peso: 0.20)
    - Proporção de outliers: 0.981 [low]
      Taxa média de outliers de 1.86% entre 6 colunas com detecção de outliers.
    - Acurácia de formato: 1.000 [low]
      0 de 4 colunas com tipo inferido possuem alta taxa de nulos, possível indicativo de dados em formato incorreto.
    - Dados suspeitos: 0.667 [medium]
      2 de 6 colunas numéricas possuem valores extremos suspeitos (além de 3× IQR).
  Completeness: 80.44 (peso: 0.25)
    - Proporção de valores não nulos: 0.931 [low]
      Média de 93.1% de valores preenchidos entre 21 colunas.
    - Completude de linhas: 0.429 [high]
      12 de 21 colunas possuem valores nulos. Score baseado na fração de colunas completamente preenchidas.
    - Proporção de strings vazias: 1.000 [low]
      Nenhuma coluna textual encontrada para avaliação.
  Consistency: 100.00 (peso: 0.20)
    - Consistência de tipos: 1.000 [low]
      0 de 21 colunas apresentam incompatibilidade en

### 3.4 Estatísticas Descritivas

In [15]:
print("\n=== Estatísticas Numéricas ===")
for s in report.stats.numeric:
    print(f"  {s.column_name}:")
    print(f"    média={s.mean:.2f}, std={s.std:.2f}")
    print(f"    min={s.min:.2f}, p25={s.q25:.2f}, p50={s.q50:.2f}, p75={s.q75:.2f}, max={s.max:.2f}")
    print(f"    assimetria={s.skewness:.3f}, curtose={s.kurtosis:.3f}")
    print()

print("\n=== Estatísticas Categóricas ===")
for s in report.stats.categorical:
    print(f"  {s.column_name}: cardinalidade={s.cardinality}, modo={s.mode}")
    print(f"    Top valores: {s.top_values[:5]}")
    print()

print("\n=== Estatísticas Temporais ===")
for s in report.stats.temporal:
    print(f"  {s.column_name}: [{s.min_date} .. {s.max_date}], {s.range_days} dias, {s.gap_count} lacunas")
print()

print("\n=== Estatísticas de Texto ===")
for s in report.stats.text:
    print(
        f"  {s.column_name}: comprimento [{s.min_length}-{s.max_length}], média {s.avg_length:.1f}, vazios {s.empty_ratio:.1%}"
    )
print()

print("\n=== Estatísticas Booleanas ===")
for s in report.stats.boolean:
    print(f"  {s.column_name}: True={s.true_count}, False={s.false_count}, ratio={s.true_ratio:.1%}")


=== Estatísticas Numéricas ===
  id:
    média=49999.50, std=28867.66
    min=0.00, p25=24999.00, p50=49004.00, p75=74999.00, max=99999.00
    assimetria=0.000, curtose=-1.200

  idade:
    média=37.77, std=11.48
    min=18.00, p25=29.00, p50=37.00, p75=45.00, max=80.00
    assimetria=0.237, curtose=-0.363

  salario:
    média=6693.38, std=5815.39
    min=1200.00, p25=2837.09, p50=4838.67, p75=8275.78, max=35000.00
    assimetria=2.194, curtose=6.010

  score:
    média=499.64, std=288.20
    min=0.00, p25=243.77, p50=493.89, p75=740.51, max=999.99
    assimetria=-0.000, curtose=-1.201

  dependentes:
    média=0.74, std=0.89
    min=0.00, p25=0.00, p50=0.00, p75=1.00, max=5.00
    assimetria=1.154, curtose=1.027

  tempo_casa_dias:
    média=1097.51, std=631.95
    min=0.00, p25=536.00, p50=1088.00, p75=1630.00, max=2190.00
    assimetria=-0.007, curtose=-1.202


=== Estatísticas Categóricas ===
  nome: cardinalidade=49, modo=Pessoa_008157
    Top valores: [('Pessoa_008157', 9), ('P

### 3.5 Distribuições

In [16]:
print("\n=== Histogramas ===")
for col_name, bins in report.distributions.histograms.items():
    print(f"  {col_name} ({len(bins)} bins):")
    total = sum(b.count for b in bins)
    for b in bins:
        pct = b.count / total * 100
        bar = "#" * int(pct / 2)
        print(f"    [{b.lower:.0f}-{b.upper:.0f}]: {bar} {b.count} ({pct:.1f}%)")
    print()

print("\n=== Frequências Categóricas ===")
for col_name, entries in report.distributions.frequencies.items():
    print(f"  {col_name}:")
    for e in entries[:5]:
        print(f"    {e.label}: {e.count}")
    print()

print("\n=== Distribuições Temporais ===")
for col_name, points in report.distributions.temporal_charts.items():
    print(f"  {col_name}:")
    for p in points[:10]:
        print(f"    {p.period}: {p.count}")


=== Histogramas ===
  id (10 bins):
    [0-10000]: ##### 10000 (10.0%)
    [10000-20000]: ##### 10000 (10.0%)
    [20000-30000]: ##### 10000 (10.0%)
    [30000-40000]: ##### 10000 (10.0%)
    [40000-50000]: ##### 10000 (10.0%)
    [50000-59999]: ##### 10000 (10.0%)
    [59999-69999]: ##### 10000 (10.0%)
    [69999-79999]: ##### 10000 (10.0%)
    [79999-89999]: ##### 10000 (10.0%)
    [89999-99999]: #### 9999 (10.0%)

  idade (10 bins):
    [18-24]: ####### 13741 (14.0%)
    [24-30]: ###### 13609 (13.9%)
    [30-37]: ######### 18329 (18.7%)
    [37-43]: ######### 19070 (19.5%)
    [43-49]: ####### 15563 (15.9%)
    [49-55]: ##### 11087 (11.3%)
    [55-61]: ## 4322 (4.4%)
    [61-68]:  1622 (1.7%)
    [68-74]:  507 (0.5%)
    [74-80]:  128 (0.1%)

  salario (10 bins):
    [1200-4580]: ####################### 43916 (46.7%)
    [4580-7960]: ############ 24373 (25.9%)
    [7960-11340]: ###### 11730 (12.5%)
    [11340-14720]: ### 5918 (6.3%)
    [14720-18100]: # 3205 (3.4%)
    [18100-21480

### 3.6 Correlações

In [17]:
print(f"Método principal: {report.correlations.method}")
print()

print("Matriz de Correlação:")
cols = sorted(report.correlations.matrix.keys())
header = "".join(f"{c:>12s}" for c in cols)
print(f"{'':>12s}{header}")
for c1 in cols:
    row = f"{c1:>12s}"
    for c2 in cols:
        row += f"{report.correlations.matrix[c1][c2]:>12.4f}"
    print(row)
print()

print("Pares com correlação > |0.5|:")
for corr in report.correlations.correlations:
    if abs(corr.value) > 0.5:
        print(f"  {corr.column_a} x {corr.column_b} = {corr.value:.4f} ({corr.method})")

Método principal: —

Matriz de Correlação:
            

Pares com correlação > |0.5|:


### 3.7 Outliers

In [18]:
print("Outliers detectados:")
for o in report.outliers.outliers:
    print(f"  {o.column_name}:")
    print(f"    Método: {o.method}")
    print(f"    Count: {o.count} ({o.ratio:.4%})")
    print(f"    Limites: [{o.bounds_lower} .. {o.bounds_upper}]")
    print()

if not report.outliers.outliers:
    print("  Nenhum outlier detectado.")

Outliers detectados:
  id:
    Método: iqr
    Count: 0 (0.0000%)
    Limites: [-50001.0 .. 149999.0]

  idade:
    Método: iqr
    Count: 383 (0.3800%)
    Limites: [5.0 .. 69.0]

  salario:
    Método: iqr
    Count: 6227 (6.2300%)
    Limites: [-5320.945 .. 16433.815]

  score:
    Método: iqr
    Count: 0 (0.0000%)
    Limites: [-501.34 .. 1485.62]

  dependentes:
    Método: iqr
    Count: 4554 (4.5500%)
    Limites: [-1.5 .. 2.5]

  tempo_casa_dias:
    Método: iqr
    Count: 0 (0.0000%)
    Limites: [-1105.0 .. 3271.0]



### 3.8 Insights

In [19]:
print(f"Total de insights: {len(report.insights.insights)}")
print()

for insight in report.insights.insights:
    col = f" [{insight.column}]" if insight.column else ""
    val = f" ({insight.metric_value:.2f})" if insight.metric_value is not None else ""
    print(f"  [{insight.severity.upper():8s}] {insight.category}{col}: {insight.message}{val}")

Total de insights: 5

  [CRITICAL] nulls [timestamp_nulo]: A coluna 'timestamp_nulo' possui 100.0% de valores nulos (100000 de 100000 registros). (1.00)
  [HIGH    ] skewness [salario]: A coluna 'salario' apresenta assimetria de 2.19 (inclinação à direita), sugerindo distribuição não normal. (2.19)
  [HIGH    ] duplicates: Taxa estimada de duplicatas de 100.0% com base no unique ratio médio de 11 colunas categóricas. (1.00)
  [MEDIUM  ] constant [coluna_constante]: A coluna 'coluna_constante' é constante (cardinalidade 1 — todos os registros possuem o mesmo valor). (1.00)
  [LOW     ] skewness [dependentes]: A coluna 'dependentes' apresenta assimetria de 1.15 (inclinação à direita), sugerindo distribuição não normal. (1.15)


### 3.9 Recomendações

In [20]:
print(f"Total de recomendações: {len(report.recommendations.recommendations)}")
print()

for rec in report.recommendations.recommendations:
    col = f" [{rec.column}]" if rec.column else ""
    print(f"  P{rec.priority} [{rec.category}]{col}")
    print(f"    Problema: {rec.message}")
    print(f"    Ação: {rec.action}")
    print()

Total de recomendações: 6

  P1 [null_treatment] [timestamp_nulo]
    Problema: Coluna 'timestamp_nulo' com 100.0% de valores nulos.
    Ação: Avaliar a causa dos nulos em 'timestamp_nulo': se o dado não está disponível, considerar preenchimento com valor default, mediana/moda, ou registro separado. Se o nulo é esperado, documentar a regra de negócio.

  P1 [null_treatment] [timestamp_nulo]
    Problema: Alta taxa de nulos em 'timestamp_nulo' pode inviabilizar análises que dependem desta coluna.
    Ação: Verificar a origem dos dados de 'timestamp_nulo': o campo é opcional na fonte? Houve falha de captura? Se possível, enriquecer com fonte alternativa.

  P2 [type_fix] [salario]
    Problema: Coluna 'salario' com distribuição assimétrica (skewness = 2.1945).
    Ação: Para modelos sensíveis a distribuição, aplicar transformação logarítmica ou Box-Cox em 'salario'. Se for análise descritiva, preferir mediana à média como medida de tendência central.

  P2 [schema]
    Problema: Taxa est

## 4. Avaliação de Qualidade Isolada

Executamos `spark_eda.assess_quality()` separadamente para comparar.

In [21]:
print("=" * 60)
print("AVALIAÇÃO DE QUALIDADE ISOLADA")
print("=" * 60)
print()

qconfig = QualityConfig(
    weights={
        "completeness": 0.30,
        "uniqueness": 0.20,
        "consistency": 0.20,
        "timeliness": 0.15,
        "accuracy": 0.15,
    },
    near_constant_threshold=0.01,
)

start = time.time()
quality_only = spark_eda.assess_quality(df, config=qconfig)
elapsed_q = time.time() - start

print(f"Avaliação concluída em {elapsed_q:.2f}s")
print(f"Score geral: {quality_only.overall:.2f} / 100.0")
print()

print("Dimensões:")
for dim in quality_only.dimensions:
    print(f"  {dim.name:15s} {dim.score:>6.2f}  (peso: {dim.weight:.2f})")
print()

print("Top Penalizadores:")
for p in quality_only.top_penalizers[:5]:
    print(f"  [{p.severity.upper():8s}] {p.name} = {p.score:.3f}")
    print(f"    {p.reason}")
    print(f"    Colunas: {', '.join(p.affected_columns)}")

AVALIAÇÃO DE QUALIDADE ISOLADA



Avaliação concluída em 30.47s
Score geral: 78.18 / 100.0

Dimensões:
  accuracy         92.87  (peso: 0.20)
  completeness     80.44  (peso: 0.25)
  consistency     100.00  (peso: 0.20)
  timeliness       71.99  (peso: 0.15)
  uniqueness       43.51  (peso: 0.20)

Top Penalizadores:
  [CRITICAL] Proporção de duplicatas = 0.000
    Unique ratio médio de 0.0% entre 11 colunas categóricas. Valores próximos de 1.0 indicam baixa duplicação.
    Colunas: nome, email, cidade, estado, categoria, documento, cep, observacao, regiao, coluna_constante, coluna_quase_constante
  [HIGH    ] Completude de linhas = 0.429
    12 de 21 colunas possuem valores nulos. Score baseado na fração de colunas completamente preenchidas.
    Colunas: nome, email, cidade, estado, idade, salario, categoria, ultimo_acesso, dependentes, documento, regiao, timestamp_nulo
  [CRITICAL] Unicidade de chaves primárias = 0.000
    0 de 2 colunas candidatas a chave primária possuem unique ratio ≥ 99%.
    Colunas: id, idade
  

## 5. Exportar Relatório HTML

Cada seção do relatório possui um método `_repr_html_()` que gera HTML formatado.
Vamos salvar tudo em um único arquivo HTML.

In [22]:
def build_html_report(report) -> str:
    """Constrói um relatório HTML completo a partir das seções."""

    sections = []

    # Título
    sections.append(f"""
    <div style="background:linear-gradient(135deg,#1a1a2e,#16213e);
                padding:32px;border-radius:12px;margin-bottom:24px;text-align:center;">
        <h1 style="color:#fff;margin:0 0 8px;font-size:28px;">Spark EDA Report</h1>
        <p style="color:#94a3b8;margin:0;font-size:14px;">
            Gerado em {datetime.now().strftime("%d/%m/%Y %H:%M")}
            | {report.overview.row_count:,} registros | {report.overview.column_count} colunas
        </p>
    </div>
    """)

    # Score de Qualidade em destaque
    score = report.quality.overall
    color = "#22c55e" if score >= 80 else ("#eab308" if score >= 50 else "#dc2626")
    sections.append(f"""
    <div style="display:flex;gap:24px;margin-bottom:24px;flex-wrap:wrap;">
        <div style="flex:1;min-width:200px;background:#f8fafc;padding:24px;border-radius:12px;
                    border:1px solid #e2e8f0;text-align:center;">
            <div style="font-size:48px;font-weight:700;color:{color};">{score:.1f}</div>
            <div style="font-size:14px;color:#64748b;margin-top:4px;">Quality Score</div>
        </div>
        <div style="flex:2;min-width:300px;background:#f8fafc;padding:24px;border-radius:12px;
                    border:1px solid #e2e8f0;">
            <div style="display:grid;grid-template-columns:1fr 1fr;gap:12px;">
    """)
    for dim in report.quality.dimensions:
        dc = "#22c55e" if dim.score >= 80 else ("#eab308" if dim.score >= 50 else "#dc2626")
        sections.append(f"""
                <div>
                    <div style="font-size:12px;color:#64748b;text-transform:uppercase;">{dim.name}</div>
                    <div style="font-size:24px;font-weight:600;color:{dc};">{dim.score:.1f}</div>
                </div>
        """)
    sections.append("""
            </div>
        </div>
    </div>
    """)

    # Sections
    section_configs = [
        ("Visão Geral", report.overview._repr_html_()),
        ("Schema", report.schema._repr_html_()),
        ("Estatísticas", report.stats._repr_html_()),
        ("Distribuições", report.distributions._repr_html_()),
        ("Correlações", report.correlations._repr_html_()),
        ("Outliers", report.outliers._repr_html_()),
        ("Insights", report.insights._repr_html_()),
        ("Recomendações", report.recommendations._repr_html_()),
    ]

    for title, html in section_configs:
        if html and "No " not in html[:100]:
            sections.append(f"""
                <div style="margin-bottom:24px;background:#fff;padding:20px;border-radius:12px;
                            border:1px solid #e2e8f0;">
                    <h2 style="font-size:18px;color:#1a1a2e;margin:0 0 16px;padding-bottom:8px;
                               border-bottom:2px solid #2563eb;">{title}</h2>
                    {html}
                </div>
            """)
        else:
            sections.append(f"""
                <div style="margin-bottom:24px;background:#fff;padding:20px;border-radius:12px;
                            border:1px solid #e2e8f0;">
                    <h2 style="font-size:18px;color:#1a1a2e;margin:0 0 16px;padding-bottom:8px;
                               border-bottom:2px solid #2563eb;">{title}</h2>
                    <p style="color:#64748b;">Nenhum dado disponível para esta seção.</p>
                </div>
            """)

    html_template = f"""<!DOCTYPE html>
<html lang="pt-BR">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Spark EDA Report</title>
    <style>
        :root {{
            --card-bg: #f8fafc;
            --border: #e2e8f0;
            --text: #1a1a2e;
            --muted: #64748b;
            --primary: #2563eb;
            --success: #16a34a;
            --warning: #d97706;
        }}
        body {{
            font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
            background: #f1f5f9;
            color: var(--text);
            margin: 0;
            padding: 24px;
            line-height: 1.5;
        }}
        .container {{
            max-width: 1200px;
            margin: 0 auto;
        }}
        table {{
            width: 100%;
            border-collapse: collapse;
        }}
        th {{
            text-align: left;
            padding: 8px 12px;
            border-bottom: 2px solid var(--border);
            font-size: 12px;
            text-transform: uppercase;
            color: var(--muted);
        }}
        td {{
            padding: 8px 12px;
            border-bottom: 1px solid var(--border);
            font-size: 13px;
        }}
    </style>
</head>
<body>
    <div class="container">
        {"".join(sections)}
        <div style="text-align:center;padding:24px;color:#94a3b8;font-size:12px;">
            Gerado por spark_eda v{spark_eda.__version__} | PySpark
        </div>
    </div>
</body>
</html>"""
    return html_template


print("Gerando relatório HTML...")
html_report = build_html_report(report)

output_path = Path("spark_eda_report.html")
output_path.write_text(html_report, encoding="utf-8")
print(f"Relatório salvo em: {output_path.resolve()}")

Gerando relatório HTML...
Relatório salvo em: /home/jovyan/work/notebooks/spark_eda_report.html


### 5.1 Preview do HTML (opcional)

No Jupyter, o relatório renderiza automaticamente com `display(report)`.
Se quiser visualizar o HTML exportado, abra o arquivo no navegador.

In [23]:
# Opcional: abrir o arquivo no navegador

html_path = Path("spark_eda_report.html").resolve()
print(f"Arquivo HTML: file:///{html_path}")

# Descomente para abrir automaticamente:
# webbrowser.open(f"file:///{html_path}")

Arquivo HTML: file:////home/jovyan/work/notebooks/spark_eda_report.html


## 6. Métricas do Pipeline

Resumo das etapas executadas e tempo de cada uma.

In [24]:
print("=" * 50)
print("RESUMO DO PIPELINE")
print("=" * 50)
print()
print(f"  Dataset: {report.overview.row_count:,} linhas x {report.overview.column_count} colunas")
print(f"  Duplicatas: {report.overview.duplicate_count:,} ({report.overview.duplicate_ratio:.2%})")
print(f"  Missing ratio: {report.overview.missing_ratio:.2%}")
print(f"  Tamanho estimado: {report.overview.size_estimate / 1024 / 1024:.1f} MB")
print()
print(f"  Quality Score: {report.quality.overall:.2f} / 100")
print(f"  Insights gerados: {len(report.insights.insights)}")
print(f"  Recomendações: {len(report.recommendations.recommendations)}")
print(f"  Correlações calculadas: {len(report.correlations.correlations)}")
print(f"  Colunas com outliers: {len(report.outliers.outliers)}")
print()
print(f"  Tempo analyze():       {elapsed:.2f}s")
print(f"  Tempo assess_quality(): {elapsed_q:.2f}s")
print()
print("Pipeline concluído com sucesso!")

RESUMO DO PIPELINE

  Dataset: 100,000 linhas x 21 colunas
  Duplicatas: 0 (0.00%)
  Missing ratio: 6.91%
  Tamanho estimado: 59.0 MB

  Quality Score: 78.18 / 100
  Insights gerados: 5
  Recomendações: 6
  Correlações calculadas: 0
  Colunas com outliers: 6

  Tempo analyze():       39.34s
  Tempo assess_quality(): 30.47s

Pipeline concluído com sucesso!


## 7. Finalização

Encerramos a SparkSession para liberar recursos.

In [25]:
spark.stop()
print("SparkSession encerrada.")

SparkSession encerrada.


---
*Notebook gerado como parte da validação da biblioteca `spark_eda`.*